# EDA MVTec Anomaly Detection Dataset

In [ ]:
# Standard libraries
import os
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

# Third-party libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from google.colab import drive

# Buoc 1: Mount Google Drive
drive.mount('/content/drive', force_remount=True)

VALID_IMAGE_EXTENSIONS: Tuple[str, ...] = (".png", ".jpg", ".jpeg", ".tiff", ".bmp")

# Buoc 2: Giai nen dataset goc tu Drive vao Colab (chi can chay 1 lan/phien)
RAW_DATASET_DIR = "/content/mvtec_raw"
DRIVE_FILE_PATH = "/content/drive/MyDrive/KLTN/mvtec_anomaly_detection.tar.xz"  # sua duoi file neu khac

os.makedirs(RAW_DATASET_DIR, exist_ok=True)

if not any(Path(RAW_DATASET_DIR).iterdir()):
    print("Dang giai nen dataset...")
    get_ipython().system('tar -xf "{DRIVE_FILE_PATH}" -C "{RAW_DATASET_DIR}"')
    print("Da giai nen xong.")
else:
    print("Dataset da duoc giai nen tu truoc, bo qua buoc giai nen.")

DATASET_PATH: Path = Path(RAW_DATASET_DIR)

if not DATASET_PATH.exists():
    print(f"[CANH BAO] Khong tim thay dataset tai: {DATASET_PATH}")
else:
    print(f"Dataset san sang tai: {DATASET_PATH}")
    print(f"Danh sach san pham: {sorted([d.name for d in DATASET_PATH.iterdir() if d.is_dir()])}")

## 1. Dataset overview: File count & Định dạng file

In [ ]:
all_files: List[Path] = [f for f in DATASET_PATH.rglob("*") if f.is_file()]

print(f"Tong so file trong dataset: {len(all_files)}")

extensions = [f.suffix.lower() for f in all_files]
ext_counts = Counter(extensions)
print("\nDinh dang file:")
for ext, count in ext_counts.items():
    print(f"- {ext if ext else 'No extension'}: {count} files")

## 2. Image count theo class & 3. Class distribution

In [ ]:
def count_images_by_class(dataset_path: Path) -> pd.DataFrame:
    """Dem so luong anh Train/Test/Ground Truth cho tung class trong dataset.

    Args:
        dataset_path: Duong dan goc toi thu muc dataset MVTec AD.

    Returns:
        DataFrame thong ke so luong anh moi class.
    """
    classes = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])
    stats = []

    for cls in classes:
        counts = {}
        for split in ("train", "test", "ground_truth"):
            split_path = dataset_path / cls / split
            imgs = [
                f for f in split_path.rglob("*")
                if f.is_file() and f.suffix.lower() in VALID_IMAGE_EXTENSIONS
            ] if split_path.exists() else []
            counts[split] = len(imgs)

        stats.append({
            "Class": cls,
            "Train Images": counts["train"],
            "Test Images": counts["test"],
            "Ground Truth Masks": counts["ground_truth"],
            "Total Images (Train+Test)": counts["train"] + counts["test"],
        })

    return pd.DataFrame(stats)


df_stats = count_images_by_class(DATASET_PATH)
print(f"Tong so classes: {len(df_stats)}\n")
display(df_stats)

## Trực quan hóa Class distribution

In [ ]:
df_melted = df_stats.melt(
    id_vars="Class",
    value_vars=["Train Images", "Test Images", "Ground Truth Masks"],
    var_name="Split",
    value_name="Count",
)

plt.figure(figsize=(14, 6))
sns.barplot(data=df_melted, x="Class", y="Count", hue="Split")
plt.xticks(rotation=45, ha="right")
plt.title("Image Count by Class and Split (Train / Test / Ground Truth)")
plt.ylabel("Number of Images / Masks")
plt.xlabel("Class")
plt.tight_layout()
plt.show()

## 4. Kiểm tra kích thước hình ảnh

In [ ]:
def check_image_sizes(dataset_path: Path) -> Dict[str, Dict[str, List[Tuple[int, int]]]]:
    """Kiem tra tinh dong nhat ve kich thuoc anh trong tung class/split cua dataset.

    Args:
        dataset_path: Duong dan goc toi thu muc dataset MVTec AD.

    Returns:
        Dictionary thong ke: {class_name: {split: [danh sach kich thuoc (W, H)]}}
    """
    classes = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])
    report: Dict[str, Dict[str, List[Tuple[int, int]]]] = {}

    for cls in classes:
        print(f"\n{'=' * 10} Class: {cls} {'=' * 10}")
        report[cls] = {}

        for split in ("train", "test", "ground_truth"):
            split_path = dataset_path / cls / split
            if not split_path.exists():
                continue

            imgs = [
                f for f in split_path.rglob("*")
                if f.is_file() and f.suffix.lower() in VALID_IMAGE_EXTENSIONS
            ]
            if not imgs:
                continue

            sizes_dict: Dict[Tuple[int, int], List[str]] = {}
            for img_path in imgs:
                try:
                    with Image.open(img_path) as img:
                        sizes_dict.setdefault(img.size, []).append(str(img_path))
                except OSError as e:
                    print(f"Loi doc anh {img_path}: {e}")

            report[cls][split] = list(sizes_dict.keys())
            print(f"  [{split.upper()}] Kich thuoc (W, H) tim thay: {list(sizes_dict.keys())}")

            if len(sizes_dict) > 1:
                print(f"  -> PHAT HIEN KICH THUOC KHONG DONG NHAT TRONG {split.upper()} CUA {cls}!")
                majority_size = max(sizes_dict, key=lambda k: len(sizes_dict[k]))
                for size, files in sizes_dict.items():
                    if size != majority_size:
                        print(f"     * File kich thuoc bat thuong {size}: {len(files)} file")
                        for f in files[:10]:
                            print(f"       - {f}")
                        if len(files) > 10:
                            print(f"       ... va {len(files) - 10} file khac.")

    return report


size_report = check_image_sizes(DATASET_PATH)

## 5. Thống kê chi tiết theo từng loại khuyết tật (Defect Type)

In [ ]:
def count_images_by_defect_type(dataset_path: Path) -> pd.DataFrame:
    """Dem so luong anh loi theo tung defect_type cu the (khong chi theo class).

    Args:
        dataset_path: Duong dan goc toi thu muc dataset MVTec AD.

    Returns:
        DataFrame thong ke: Class, Defect Type, So anh.
    """
    classes = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])
    stats: List[Dict[str, object]] = []

    for cls in classes:
        test_path = dataset_path / cls / "test"
        if not test_path.exists():
            continue

        defect_dirs = sorted([d for d in test_path.iterdir() if d.is_dir() and d.name != "good"])
        for defect_dir in defect_dirs:
            n_images = len([
                f for f in defect_dir.iterdir()
                if f.is_file() and f.suffix.lower() in VALID_IMAGE_EXTENSIONS
            ])
            stats.append({
                "Class": cls,
                "Defect Type": defect_dir.name,
                "So anh": n_images,
            })

    return pd.DataFrame(stats)


df_defect_stats = count_images_by_defect_type(DATASET_PATH)
print(f"Tong so loai khuyet tat: {len(df_defect_stats)}")
print(f"Tong so anh loi: {df_defect_stats['So anh'].sum()}")
display(df_defect_stats)

## 6. Phân tích diện tích vùng lỗi trong Mask (Ground Truth)

In [ ]:
import cv2
import numpy as np

MIN_COMPONENT_AREA_PX: int = 5  # nguong loc nhieu toi thieu, chi de thong ke tho


def analyze_defect_areas(dataset_path: Path) -> np.ndarray:
    """Phan tich dien tich cac vung loi (connected components) trong toan bo mask.

    Args:
        dataset_path: Duong dan goc toi thu muc dataset MVTec AD.

    Returns:
        Mang numpy chua dien tich (so pixel) cua tung vung loi tim duoc.
    """
    areas: List[int] = []
    classes = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])

    for cls in classes:
        gt_path = dataset_path / cls / "ground_truth"
        if not gt_path.exists():
            continue

        for mask_path in gt_path.rglob("*"):
            if not mask_path.is_file() or mask_path.suffix.lower() not in VALID_IMAGE_EXTENSIONS:
                continue

            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue

            _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
            num_labels, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

            for label_id in range(1, num_labels):
                area = stats[label_id, cv2.CC_STAT_AREA]
                if area >= MIN_COMPONENT_AREA_PX:
                    areas.append(int(area))

    return np.array(areas)


defect_areas = analyze_defect_areas(DATASET_PATH)

print(f"Tong so vung loi tim duoc: {len(defect_areas)}")
print(f"Dien tich nho nhat : {defect_areas.min()} px")
print(f"Dien tich trung vi : {int(np.median(defect_areas))} px")
print(f"Dien tich lon nhat : {defect_areas.max()} px")
print(f"Percentile 2%      : {int(np.percentile(defect_areas, 2))} px")

plt.figure(figsize=(10, 5))
plt.hist(defect_areas, bins=60, color="indianred")
plt.xlabel("Dien tich vung loi (so pixel)")
plt.ylabel("So luong vung loi")
plt.title("Phan bo dien tich vung loi trong toan bo dataset")
plt.yscale("log")
plt.tight_layout()
plt.show()

## 7. Hiển thị ảnh mẫu (Ảnh gốc | Mask | Ảnh chồng Mask)

In [ ]:
import random

random.seed(42)


def visualize_sample_defects(dataset_path: Path, n_samples: int = 4) -> None:
    """Hien thi luoi anh mau: anh goc, mask, va anh chong mask mau do.

    Args:
        dataset_path: Duong dan goc toi thu muc dataset MVTec AD.
        n_samples: So luong class ngau nhien duoc chon de minh hoa.
    """
    classes = [d for d in dataset_path.iterdir() if d.is_dir()]
    sample_classes = random.sample(classes, min(n_samples, len(classes)))

    fig, axes = plt.subplots(len(sample_classes), 3, figsize=(11, 3.3 * len(sample_classes)))
    if len(sample_classes) == 1:
        axes = [axes]

    for row, cls_dir in enumerate(sample_classes):
        gt_dir = cls_dir / "ground_truth"
        defect_dirs = [d for d in gt_dir.iterdir() if d.is_dir()] if gt_dir.exists() else []
        if not defect_dirs:
            continue

        defect_dir = random.choice(defect_dirs)
        mask_files = list(defect_dir.glob("*"))
        if not mask_files:
            continue

        mask_path = random.choice(mask_files)
        img_path = cls_dir / "test" / defect_dir.name / mask_path.name.replace("_mask", "")

        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        overlay = img.copy()
        overlay[mask > 127] = [255, 0, 0]
        blended = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)

        axes[row][0].imshow(img)
        axes[row][0].set_title(f"{cls_dir.name}/{defect_dir.name}\n(goc)", fontsize=9)
        axes[row][0].axis("off")

        axes[row][1].imshow(mask, cmap="gray")
        axes[row][1].set_title("mask", fontsize=9)
        axes[row][1].axis("off")

        axes[row][2].imshow(blended)
        axes[row][2].set_title("chong mask (do)", fontsize=9)
        axes[row][2].axis("off")

    plt.tight_layout()
    plt.show()


visualize_sample_defects(DATASET_PATH, n_samples=4)

## 8. Kiểm tra tính toàn vẹn dữ liệu (Missing / Empty Mask)

In [ ]:
def check_data_integrity(dataset_path: Path) -> Tuple[List[str], List[str]]:
    """Kiem tra anh thieu mask tuong ung va mask rong (khong co pixel loi).

    Args:
        dataset_path: Duong dan goc toi thu muc dataset MVTec AD.

    Returns:
        Tuple gom (danh sach anh thieu mask, danh sach mask rong).
    """
    missing_mask: List[str] = []
    empty_mask: List[str] = []
    classes = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])

    for cls in classes:
        test_path = dataset_path / cls / "test"
        gt_path = dataset_path / cls / "ground_truth"
        if not test_path.exists():
            continue

        for defect_dir in test_path.iterdir():
            if not defect_dir.is_dir() or defect_dir.name == "good":
                continue

            for img_path in defect_dir.glob("*"):
                if img_path.suffix.lower() not in VALID_IMAGE_EXTENSIONS:
                    continue

                mask_path = gt_path / defect_dir.name / f"{img_path.stem}_mask.png"
                if not mask_path.exists():
                    missing_mask.append(str(img_path))
                    continue

                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                if mask is not None and mask.max() == 0:
                    empty_mask.append(str(mask_path))

    return missing_mask, empty_mask


missing_masks, empty_masks = check_data_integrity(DATASET_PATH)

print(f"So anh THIEU mask tuong ung: {len(missing_masks)}")
if missing_masks:
    print("  Vi du:", missing_masks[:5])

print(f"So mask RONG (khong co pixel loi): {len(empty_masks)}")
if empty_masks:
    print("  Vi du:", empty_masks[:5])

if not missing_masks and not empty_masks:
    print("\n>>> DU LIEU SACH. San sang chuyen sang tien xu ly.")